In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/diabetes_data/diabetic_data.csv")
df = df.replace('?', np.nan)
df = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])
df['race'] = df['race'].fillna('Unknown')

df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset='patient_nbr', keep='first')

exclude_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(exclude_ids)]

df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)
df.shape

(69973, 48)

In [8]:
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

categorical_features = [
    'race', 'gender', 'age', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id',
    'A1Cresult', 'max_glu_serum', 'change', 'diabetesMed', 'insulin'
]

X = df[numeric_features + categorical_features]
y = df['readmitted_binary']

X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
X.shape

(69973, 76)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((55978, 76), (13995, 76))

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)                      # changed

y_pred = model.predict(X_test_scaled)                    # changed
y_proba = model.predict_proba(X_test_scaled)[:, 1]       # changed

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.94      0.68      0.79     12740
           1       0.14      0.52      0.22      1255

    accuracy                           0.67     13995
   macro avg       0.54      0.60      0.51     13995
weighted avg       0.86      0.67      0.74     13995

ROC-AUC: 0.6421338820542007


In [14]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_tree))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tree))

              precision    recall  f1-score   support

           0       0.93      0.60      0.73     12740
           1       0.12      0.57      0.20      1255

    accuracy                           0.60     13995
   macro avg       0.53      0.59      0.47     13995
weighted avg       0.86      0.60      0.68     13995

ROC-AUC: 0.6096696416844395


Baseline comparison: logistic regression (ROC-AUC 0.64) outperformed a depth-5 decision tree (ROC-AUC 0.61); logistic regression selected as this week's best baseline.